# Chapter 1 Source Code Notebook

This notebook contains the source code for Chapter 1, **Beyond LLM Centrism, Intelligence as a System**.

The code builds the SupportOps AI baseline in four parts:

1. The naive LLM centric support handler
2. Cost and latency tracking
3. A modular refactor with explicit state, classification, deterministic routing, and response drafting
4. A comparison runner for the modular approach

Before running the notebook, install the `anthropic` package and set the `ANTHROPIC_API_KEY` environment variable.

## Step 1: The Naive LLM Centric Version

This first version keeps everything in one model call. The model classifies the ticket, assesses urgency, decides whether to escalate, drafts the response, and suggests next steps.

In [ ]:
# Import the Anthropic SDK for model access.
import anthropic

# Import time because later cells use timing for latency tracking.
import time

# Create the Anthropic client.
# This expects ANTHROPIC_API_KEY to be available in your environment.
client = anthropic.Anthropic()

# System prompt used by the naive support handler.
# This prompt asks the model to perform classification, escalation, response drafting, and next step generation.
SYSTEM_PROMPT = """You are a customer support AI for SupportOps.
When you receive a customer ticket, you must:
1. Classify the issue (billing, technical, account, other)
2. Assess urgency (low, medium, high, critical)
3. Determine whether to escalate (yes/no) based on urgency and content
4. Draft a response to the customer
5. Suggest next steps for the support team

Return policy is 30 days. Standard response time is 24 hours.
Always be professional and empathetic."""


def handle_ticket_naive(ticket_text: str) -> str:
    # Send the ticket text to the model using the full system prompt.
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": ticket_text}]
    )

    # Return the text content from the first response block.
    return response.content[0].text


## Step 2: Adding the Cost and Latency Tracker

This tracker records token usage, estimated cost, and latency for each model call. It makes the cost and performance behavior of the system visible per component.

In [ ]:
# Import the Anthropic SDK for model calls.
import anthropic

# Import time for latency measurement.
import time

# Import json for structured response parsing in later cells.
import json

# Import dataclass helpers for trace objects.
from dataclasses import dataclass, field

# Import typing helpers used by the trace dataclasses.
from typing import List, Optional

# Token costs per million tokens.
# Adjust these values to match your actual model pricing.
COST_PER_MILLION_INPUT = 3.00
COST_PER_MILLION_OUTPUT = 15.00


@dataclass
class RequestTrace:
    # Unique identifier for this component call within the trace.
    request_id: str

    # Name of the component that made the model call.
    component: str

    # Number of input tokens reported by the model provider.
    input_tokens: int = 0

    # Number of output tokens reported by the model provider.
    output_tokens: int = 0

    # Request latency in milliseconds.
    latency_ms: float = 0.0

    # Estimated request cost in US dollars.
    cost_usd: float = 0.0


@dataclass
class SystemTrace:
    # Ticket identifier used to group component traces.
    ticket_id: str

    # List of individual request traces for this ticket.
    traces: List[RequestTrace] = field(default_factory=list)

    @property
    def total_cost(self) -> float:
        # Sum the estimated cost across all component calls.
        return sum(t.cost_usd for t in self.traces)

    @property
    def total_latency_ms(self) -> float:
        # Sum the latency across all component calls.
        return sum(t.latency_ms for t in self.traces)

    @property
    def total_tokens(self) -> int:
        # Sum input and output tokens across all component calls.
        return sum(t.input_tokens + t.output_tokens for t in self.traces)


def tracked_call(client, component, trace, **kwargs):
    # Capture the request start time.
    start = time.time()

    # Make the model call with the supplied keyword arguments.
    response = client.messages.create(**kwargs)

    # Convert elapsed time to milliseconds.
    latency = (time.time() - start) * 1000

    # Read token usage from the provider response.
    input_tok = response.usage.input_tokens
    output_tok = response.usage.output_tokens

    # Estimate the cost using input and output token prices.
    cost = (
        (input_tok / 1_000_000) * COST_PER_MILLION_INPUT +
        (output_tok / 1_000_000) * COST_PER_MILLION_OUTPUT
    )

    # Add this component call to the system trace.
    trace.traces.append(RequestTrace(
        request_id=f"{component}-{len(trace.traces)+1}",
        component=component,
        input_tokens=input_tok,
        output_tokens=output_tok,
        latency_ms=latency,
        cost_usd=cost
    ))

    # Return the original model response unchanged.
    return response


## Step 3: The Modular Refactor

The modular version separates language tasks from deterministic logic. The model handles classification and response drafting, while Python handles state, escalation rules, and routing.

In [ ]:
# Import json for parsing structured model output.
import json

# Import dataclass for structured state objects.
from dataclasses import dataclass

# Import Enum for controlled category and urgency values.
from enum import Enum


class IssueCategory(Enum):
    BILLING = "billing"
    TECHNICAL = "technical"
    ACCOUNT = "account"
    OTHER = "other"


class UrgencyLevel(Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"


@dataclass
class TicketClassification:
    # Classified issue category.
    category: IssueCategory

    # Classified urgency level.
    urgency: UrgencyLevel

    # Customer sentiment label.
    sentiment: str  # positive, neutral, negative, hostile

    # Specific factual claims made by the customer.
    key_claims: list[str]

    # Whether deterministic rules should require human review.
    requires_human_review: bool


@dataclass
class TicketState:
    # Ticket identifier.
    ticket_id: str

    # Original customer message.
    raw_text: str

    # Structured classification returned by the model and validated by Python.
    classification: TicketClassification | None = None

    # Whether this ticket has been escalated.
    escalated: bool = False

    # Reason for escalation when escalation occurs.
    escalation_reason: str = ""

    # Draft response generated for the customer.
    draft_response: str = ""

    # Queue selected by deterministic routing logic.
    routing_queue: str = "general"

    # Processing status for the ticket.
    status: str = "open"


### Classification Function

This function asks the model to perform a bounded language task and return structured JSON. Python then parses and validates that JSON into typed objects.

In [ ]:
# Prompt used for ticket classification.
# The model is asked to return only valid JSON so the system can parse and validate it.
CLASSIFICATION_PROMPT = """Analyze this support ticket and return a JSON object with:
- category: one of [billing, technical, account, other]
- urgency: one of [low, medium, high, critical]
- sentiment: one of [positive, neutral, negative, hostile]
- key_claims: list of specific factual claims the customer is making
- requires_human_review: true if the ticket contains legal threats,
  personal safety concerns, or regulatory language

Return ONLY valid JSON. No preamble, no explanation."""


def classify_ticket(
    client, ticket_text: str, trace: SystemTrace
) -> TicketClassification:
    # Use the tracked model call wrapper so classification cost and latency are recorded.
    response = tracked_call(
        client,
        component="classifier",
        trace=trace,
        model="claude-haiku-4-5-20251001",  # cheaper model for classification
        max_tokens=512,
        messages=[{
            "role": "user",
            "content": f"{CLASSIFICATION_PROMPT}\n\nTicket:\n{ticket_text}"
        }]
    )

    # Extract raw model text and parse it as JSON.
    raw = response.content[0].text.strip()
    data = json.loads(raw)  # explicit validation, will raise on bad output

    # Convert validated JSON fields into typed Python objects.
    return TicketClassification(
        category=IssueCategory(data["category"]),
        urgency=UrgencyLevel(data["urgency"]),
        sentiment=data["sentiment"],
        key_claims=data.get("key_claims", []),
        requires_human_review=data.get("requires_human_review", False)
    )


### Escalation and Routing Rules

This part is pure Python. Escalation and routing are deterministic, testable, and do not require a model call.

In [ ]:
# Escalation rules by urgency level.
# Critical and high urgency tickets are escalated.
ESCALATION_RULES = {
    UrgencyLevel.CRITICAL: True,
    UrgencyLevel.HIGH: True,
    UrgencyLevel.MEDIUM: False,
    UrgencyLevel.LOW: False,
}


def apply_escalation_rules(state: TicketState) -> TicketState:
    """Pure business logic, no model call needed."""
    # Read the classification from state.
    clf = state.classification

    # If there is no classification yet, return the state unchanged.
    if clf is None:
        return state

    # Human review flags always trigger escalation.
    if clf.requires_human_review:
        state.escalated = True
        state.escalation_reason = "human_review_required"

    # Urgency based escalation is handled by deterministic rules.
    elif ESCALATION_RULES.get(clf.urgency, False):
        state.escalated = True
        state.escalation_reason = f"urgency_{clf.urgency.value}"

    # Routing is also deterministic.
    state.routing_queue = {
        IssueCategory.BILLING: "billing-team",
        IssueCategory.TECHNICAL: "tech-support",
        IssueCategory.ACCOUNT: "account-management",
        IssueCategory.OTHER: "general",
    }.get(clf.category, "general")

    # Return the updated state object.
    return state


### Response Drafting

This function returns to the model for a language task, but the prompt is bounded by explicit system state. The model drafts the message without owning the escalation or routing decision.

In [ ]:
def draft_response(
    client, state: TicketState, trace: SystemTrace
) -> str:
    # Build explicit context from validated system state.
    context_parts = [
        f"Issue category: {state.classification.category.value}",
        f"Urgency: {state.classification.urgency.value}",
        f"Customer sentiment: {state.classification.sentiment}",
    ]

    # Add escalation context only when deterministic rules have escalated the ticket.
    if state.escalated:
        context_parts.append(
            f"This ticket has been escalated: {state.escalation_reason}"
        )
        context_parts.append(
            "Your response should acknowledge escalation to a specialist."
        )

    # Build the response drafting prompt.
    prompt = f"""Write a professional, empathetic customer support response.

Context:
{chr(10).join(context_parts)}

Customer message:
{state.raw_text}

Keep the response concise and actionable. Do not make specific promises about
timelines or outcomes - say you are looking into it."""

    # Use the tracked call wrapper so response drafting cost and latency are recorded.
    response = tracked_call(
        client,
        component="response_drafter",
        trace=trace,
        model="claude-sonnet-4-6",
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )

    # Return the generated response text.
    return response.content[0].text


### Full Orchestration Function

This function ties the modular pieces together. It classifies the ticket, applies deterministic rules, drafts the response, and returns both the final state and the trace.

In [ ]:
def handle_ticket_modular(
    client,
    ticket_id: str,
    ticket_text: str
) -> tuple[TicketState, SystemTrace]:
    # Create a trace object for this ticket.
    trace = SystemTrace(ticket_id=ticket_id)

    # Create the initial ticket state.
    state = TicketState(ticket_id=ticket_id, raw_text=ticket_text)

    # Step 1: Classify the ticket using a model call because this is a language task.
    state.classification = classify_ticket(client, ticket_text, trace)

    # Step 2: Apply business rules in Python because this is deterministic logic.
    state = apply_escalation_rules(state)

    # Step 3: Draft a response using a model call because this is a language task.
    state.draft_response = draft_response(client, state, trace)

    # Mark the state as processed.
    state.status = "processed"

    # Return the final state and trace for inspection.
    return state, trace


## Step 4: Comparing the Two Approaches

The final cell runs the modular version against a sample billing ticket and prints the final state, draft response, cost, and latency trace.

In [ ]:
if __name__ == "__main__":
    # Import os to read the API key from the environment.
    import os

    # Create the Anthropic client from the ANTHROPIC_API_KEY environment variable.
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

    # Sample customer ticket for the modular support workflow.
    ticket = """
    I've been charged twice for my subscription this month.
    I contacted support last week and was told it would be fixed,
    but I'm still seeing the double charge. This is completely
    unacceptable and I'm considering a chargeback if not resolved today.
    """

    # Run the modular version.
    state, trace = handle_ticket_modular(client, "TKT-001", ticket)

    # Print the processed ticket state.
    print("=== TICKET STATE ===")
    print(f"Category: {state.classification.category.value}")
    print(f"Urgency: {state.classification.urgency.value}")
    print(f"Escalated: {state.escalated} ({state.escalation_reason})")
    print(f"Queue: {state.routing_queue}")
    print(f"\nDraft Response:\n{state.draft_response}")

    # Print the per component cost and latency trace.
    print("\n=== COST AND LATENCY TRACE ===")
    for t in trace.traces:
        print(f"{t.component}: {t.input_tokens}+{t.output_tokens} tokens, "),
        print(f"${t.cost_usd:.4f}, {t.latency_ms:.0f}ms")

    # Print the total cost and latency for the request.
    print(f"\nTOTAL: ${trace.total_cost:.4f}, {trace.total_latency_ms:.0f}ms")
